# Diagnostic : les modèles sont-ils bridés ?

**Question** : la règle de surapprentissage (écart relatif train - validation de 5 % au plus) simplifie fortement les modèles : RandomForest finit à une profondeur de 2 à 4. Le plateau des scénarios (score décisionnel F2 d'environ 0,48 à 0,49) vient-il de ce bridage, ou des données ?

**Méthode** : RandomForest, CatBoost et le MLP sont **poussés dans leurs retranchements** (arbres profonds et petites feuilles, réseau plus grand et peu régularisé), sur **toutes les colonnes** du scénario complet `ml_11`, avec les **mêmes 5 plis**. Pour chaque réglage : précision des défauts prédits en validation et sur l'entraînement (au taux de rappel minimal de 60 %), écart relatif, puis score décisionnel F2 comparé **pli par pli** à `ml_11` (même modèle, et meilleur modèle).

**Ce n'est pas un scénario** : ces réglages violent volontairement la règle de surapprentissage. Rien n'est écrit dans le fichier de résultats, aucun modèle n'est enregistré, le test n'est pas lu. Les réglages sont fixés à l'avance, sans boucle.

**Lecture** : si la validation monte nettement malgré un gros écart, la règle bride trop ces modèles (à revoir avant l'élargissement des réglages). Si elle stagne ou baisse, le plateau vient des données.

## 1. Données, variables et pipeline : identiques à `ml_11`

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

train = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv")
print(f"Clients du train : {len(train)} ; colonnes disponibles : {train.shape[1]}")

Clients du train : 19358 ; colonnes disponibles : 56


In [2]:
# ==============================================================================
# VARIABLES DU SCÉNARIO : seule cellule à modifier pour changer les colonnes
# ==============================================================================

# Colonnes ajoutées par ce scénario, rangées selon leur encodage
ajout_num = ["ratio_PAY_BILL_global", "ratio_PAY_BILL_median", "ratio_PAY_BILL_regularite", "ratio_PAY_BILL1", "ratio_PAY_BILL2", "ratio_PAY_BILL3", "ratio_PAY_BILL4", "ratio_PAY_BILL5", "ratio_BILL_LIMIT1", "ratio_BILL_LIMIT2", "ratio_BILL_LIMIT3", "ratio_BILL_LIMIT4", "ratio_BILL_LIMIT5", "ratio_BILL_LIMIT6", "NB_MOIS_CTX", "CUMUL_INCIDENT"]   # nombres, montants, ratios, compteurs
ajout_log = ["BILL_evol_relatif", "PAYAMT_evol_relatif"]   # montants positifs ou nuls très étirés : logarithme, puis mise à l'échelle
ajout_log_signe = ["BILL_evol_brut", "PAYAMT_evol_brut"]   # montants qui peuvent être négatifs (avoirs) : logarithme signé, puis mise à l'échelle
ajout_cat = ["FLAG_OUVERTURE", "FLAG_DEGEL", "MOIS_ACTIVATION", "TYPE_USAGE", "PAY_habituel", "FAUX_CODAGE", "SURVEILLANCE_RECENTE", "FLAG_CTX", "MOIS_SORTIE_CTX", "FLAG_RETARD", "MOIS_SORTIE_RETARD"]   # catégories sans ordre (une colonne par catégorie), indicateurs 0 / 1
ajout_ord = []   # catégories dans un ordre naturel (encodage ordinal)

# Colonnes du socle retirées par ce scénario
retirees = []

# ------------------------------------------------------------------------------
# Socle : les 23 variables d'origine (ne pas modifier)
# PAY_n : des nombres déjà ordonnés et régulièrement espacés (-2 à 8, un cran par mois de retard), mis à l'échelle comme les autres (v6)
# PAY_AMTn : paiements très étirés (médiane d'environ 2 000 NT$, maximum de plus de 400 000 NT$), passés au logarithme (v7)
# BILL_AMTn : factures, traitées comme les paiements qu'elles appellent, avec un logarithme signé (avoirs négatifs) (v8)
socle_num = ["LIMIT_BAL", "AGE"] + [f"PAY_{n}" for n in range(1, 7)]
socle_log = [f"PAY_AMT{n}" for n in range(1, 7)]
socle_log_signe = [f"BILL_AMT{n}" for n in range(1, 7)]
socle_cat = ["SEX", "EDUCATION", "MARRIAGE"]
socle_ord = []

# ------------------------------------------------------------------------------
# Remplacements décidés dans docs/colonnes_creees.md (colonne « Adaptation et usage ML ») : {colonne: (valeurs remplacées, valeur)}
# Faits dans le pipeline, avant la transformation propre au type de la colonne (mise à l'échelle, logarithme, catégories…),
# et donc enregistrés avec le modèle. Valeur précise décidée par le métier, jamais calculée sur les clients (ni médiane ni moyenne).
# Actifs seulement si la colonne est une variable du scénario, quelle que soit sa liste. Un vide s'écrit np.nan ; une catégorie
# s'écrit en texte. À compléter à chaque nouvelle décision
remplacements = {
    "ratio_PAY_BILL_global": ([np.nan], 100),          # dû nul ou négatif : rien à payer, client à jour
    "ratio_PAY_BILL_median": ([-1], 100),              # seule la facture de septembre est due
    "ratio_PAY_BILL_regularite": ([np.nan, -1], 100),  # une seule facture due, ou « Première facture »
}

num_col = [c for c in socle_num if c not in retirees] + ajout_num
log_col = [c for c in socle_log if c not in retirees] + ajout_log
log_signe_col = [c for c in socle_log_signe if c not in retirees] + ajout_log_signe
cat_col = [c for c in socle_cat if c not in retirees] + ajout_cat
ord_col = [c for c in socle_ord if c not in retirees] + ajout_ord
variables = num_col + log_col + log_signe_col + cat_col + ord_col

# Contrôles : colonnes présentes dans le train, aucune en double, ni identifiant ni cible
absentes = [c for c in variables + retirees if c not in train.columns]
doublons = sorted({c for c in variables if variables.count(c) > 1})
interdites = [c for c in variables if c in ("ID", "dpnm")]
assert not absentes, f"Colonnes absentes du train : {absentes} (à créer dans creation_datasets_ML.ipynb)"
assert not doublons, f"Colonnes en double : {doublons}"
assert not interdites, f"Colonnes interdites comme variables : {interdites}"
# Logarithme : montants positifs ou nuls, une fois les valeurs remplacées mises de côté
for c in log_col:
    valeurs = train[c].dropna()
    if c in remplacements:
        valeurs = valeurs[~valeurs.isin([v for v in remplacements[c][0] if not pd.isna(v)])]
    assert (valeurs >= 0).all(), f"Le logarithme ne s'applique qu'à des montants positifs ou nuls : {c}"
# Remplacements : aucune variable ne garde un vide sans remplacement décidé
remplacees = [c for c in variables if c in remplacements]
vides_sans_remplacement = [c for c in variables if train[c].isna().any()
                           and not (c in remplacements and any(pd.isna(v) for v in remplacements[c][0]))]
assert not vides_sans_remplacement, f"Colonnes avec des vides, sans remplacement décidé dans docs/colonnes_creees.md : {vides_sans_remplacement}"

X_train = train[variables].reset_index(drop=True)
y_train = train["dpnm"].reset_index(drop=True)

print(f"Variables        : {len(variables)} ({len(num_col)} numériques, {len(log_col)} au logarithme, {len(log_signe_col)} au logarithme signé, {len(cat_col)} catégorielles, {len(ord_col)} ordinales)")
print(f"Ajoutées         : {ajout_num + ajout_log + ajout_log_signe + ajout_cat + ajout_ord if ajout_num + ajout_log + ajout_log_signe + ajout_cat + ajout_ord else 'aucune'}")
print(f"Retirées         : {retirees if retirees else 'aucune'}")
print(f"Remplacements    : {', '.join(f'{c} ({remplacements[c][0]} -> {remplacements[c][1]})' for c in remplacees) if remplacees else 'aucun'}")
print(f"Défauts          : {y_train.sum()} ({y_train.mean() * 100:.2f} %)")

Variables        : 54 (24 numériques, 8 au logarithme, 8 au logarithme signé, 14 catégorielles, 0 ordinales)
Ajoutées         : ['ratio_PAY_BILL_global', 'ratio_PAY_BILL_median', 'ratio_PAY_BILL_regularite', 'ratio_PAY_BILL1', 'ratio_PAY_BILL2', 'ratio_PAY_BILL3', 'ratio_PAY_BILL4', 'ratio_PAY_BILL5', 'ratio_BILL_LIMIT1', 'ratio_BILL_LIMIT2', 'ratio_BILL_LIMIT3', 'ratio_BILL_LIMIT4', 'ratio_BILL_LIMIT5', 'ratio_BILL_LIMIT6', 'NB_MOIS_CTX', 'CUMUL_INCIDENT', 'BILL_evol_relatif', 'PAYAMT_evol_relatif', 'BILL_evol_brut', 'PAYAMT_evol_brut', 'FLAG_OUVERTURE', 'FLAG_DEGEL', 'MOIS_ACTIVATION', 'TYPE_USAGE', 'PAY_habituel', 'FAUX_CODAGE', 'SURVEILLANCE_RECENTE', 'FLAG_CTX', 'MOIS_SORTIE_CTX', 'FLAG_RETARD', 'MOIS_SORTIE_RETARD']
Retirées         : aucune
Remplacements    : ratio_PAY_BILL_global ([nan] -> 100), ratio_PAY_BILL_median ([-1] -> 100), ratio_PAY_BILL_regularite ([nan, -1] -> 100)
Défauts          : 3084 (15.93 %)


In [3]:
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
import numpy as np
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# Transformation de chaque type de colonne (un nouvel objet à chaque appel)
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
    "ord": lambda: make_pipeline(OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), StandardScaler()),
}
COLONNES_PAR_TYPE = {"num": num_col, "log": log_col, "log_signe": log_signe_col, "nom": cat_col, "ord": ord_col}

def avec_remplacement(colonne, transformation):
    # Valeurs remplacées par la valeur décidée (cellule « variables du scénario »), puis transformation propre au type
    valeurs, valeur = remplacements[colonne]
    # Le vide d'abord : SimpleImputer refuse une colonne qui contient des vides quand il cherche une autre valeur
    valeurs = sorted(valeurs, key=lambda v: not pd.isna(v))
    return make_pipeline(*[SimpleImputer(missing_values=v, strategy="constant", fill_value=valeur) for v in valeurs], transformation)

# Une branche par type pour les colonnes sans remplacement, une branche par colonne avec remplacement.
# Sans remplacement (le socle), le pipeline est exactement celui de la v8
preprocessor = ColumnTransformer(transformers=
    [(nom, TRANSFORMATIONS[nom](), [c for c in colonnes if c not in remplacements]) for nom, colonnes in COLONNES_PAR_TYPE.items()]
    + [(f"remplacement_{c}", avec_remplacement(c, TRANSFORMATIONS[nom]()), [c])
       for nom, colonnes in COLONNES_PAR_TYPE.items() for c in colonnes if c in remplacements])

# Objectif de détection : part des défauts que le modèle doit trouver (60 % pour commencer, objectif 70 %)
RAPPEL_MINIMAL = 0.60

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    # Seuil le plus haut qui détecte au moins la part voulue des défauts
    proba_defauts = np.sort(proba[np.asarray(y) == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

def score_decision(estimator, X, y):
    # Précision au seuil qui détecte au moins RAPPEL_MINIMAL des défauts (score optimisé par la boucle)
    proba = estimator.predict_proba(X)[:, 1]
    signales = proba >= seuil_pour_rappel(y, proba)
    return np.asarray(y)[signales].mean()

scorers = {
    "decision": score_decision,
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles lancés dans ce notebook : retirer un nom pour ne pas le lancer (le reste du notebook s'adapte)
MODELES_A_TESTER = ["LogisticRegression", "SVM", "MLPClassifier", "KNN", "RandomForest", "CatBoost"]

def modeles_de_base():
    # Les 6 modèles d'origine, avec leurs réglages de la première itération ; seuls ceux de MODELES_A_TESTER sont rendus
    tous = {
        "LogisticRegression": make_pipeline(preprocessor, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "SVM": make_pipeline(preprocessor, CalibratedClassifierCV(estimator=SVC(class_weight="balanced", random_state=42), ensemble=False)),
        "MLPClassifier": make_pipeline(preprocessor, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "KNN": make_pipeline(preprocessor, KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
        "RandomForest": make_pipeline(preprocessor, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(preprocessor, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

## 2. Réglages poussés, fixés à l'avance

- **RandomForest** : 500 arbres, profondeur libre ou très grande, feuilles de 1 à 20 clients.
- **CatBoost** : profondeur 8 ou 10 (au lieu de 4 à 6), régularisation faible (`l2_leaf_reg = 1`), beaucoup d'arbres avec un pas d'apprentissage lent.
- **MLP** (réseau de neurones) : trois couches de 256, 128 et 64 neurones (au lieu de 50 et 25), régularisation faible (`alpha`), avec ou sans arrêt anticipé (sans, il apprend jusqu'à mémoriser l'entraînement).

La cellule affiche une ligne par réglage dès qu'il est terminé, avec sa durée.

In [4]:
from sklearn.metrics import fbeta_score
import time

REGLAGES_POUSSES = {
    "RandomForest profondeur libre, feuilles de 1": ("RandomForest", RandomForestClassifier(n_estimators=500, max_depth=None, min_samples_leaf=1, max_features="sqrt", class_weight="balanced", random_state=42, n_jobs=-1)),
    "RandomForest profondeur 20, feuilles de 5": ("RandomForest", RandomForestClassifier(n_estimators=500, max_depth=20, min_samples_leaf=5, max_features="sqrt", class_weight="balanced", random_state=42, n_jobs=-1)),
    "RandomForest profondeur 12, feuilles de 20": ("RandomForest", RandomForestClassifier(n_estimators=500, max_depth=12, min_samples_leaf=20, max_features="sqrt", class_weight="balanced", random_state=42, n_jobs=-1)),
    "CatBoost profondeur 8": ("CatBoost", CatBoostClassifier(depth=8, l2_leaf_reg=1, iterations=1500, learning_rate=0.03, scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    "CatBoost profondeur 10": ("CatBoost", CatBoostClassifier(depth=10, l2_leaf_reg=1, iterations=1000, learning_rate=0.03, scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    "MLP 256-128-64, sans arrêt anticipé": ("MLPClassifier", MLPClassifier(hidden_layer_sizes=(256, 128, 64), alpha=1e-5, early_stopping=False, max_iter=300, random_state=42)),
    "MLP 256-128-64, avec arrêt anticipé": ("MLPClassifier", MLPClassifier(hidden_layer_sizes=(256, 128, 64), alpha=1e-4, early_stopping=True, max_iter=500, random_state=42)),
    "MLP 128-64, sans arrêt anticipé": ("MLPClassifier", MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-5, early_stopping=False, max_iter=300, random_state=42)),
}

y = y_train.to_numpy()
plis = list(cv.split(X_train, y_train))   # les mêmes 5 plis que tous les scénarios

def precision_au_rappel(y_vrai, proba):
    return y_vrai[proba >= seuil_pour_rappel(y_vrai, proba)].mean()

def f2_au_rappel_minimal(y_vrai, proba):
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

mesures = {}
for nom, (modele_ml_11, estimateur) in REGLAGES_POUSSES.items():
    debut = time.time()
    prec_val, prec_train, f2_val = [], [], []
    for entrainement, validation in plis:
        pipeline = make_pipeline(preprocessor, estimateur)
        pipeline.fit(X_train.iloc[entrainement], y[entrainement])
        p_val = pipeline.predict_proba(X_train.iloc[validation])[:, 1]
        p_train = pipeline.predict_proba(X_train.iloc[entrainement])[:, 1]
        prec_val.append(precision_au_rappel(y[validation], p_val))
        prec_train.append(precision_au_rappel(y[entrainement], p_train))
        f2_val.append(f2_au_rappel_minimal(y[validation], p_val))
    mesures[nom] = {"modele_ml_11": modele_ml_11, "prec_val": prec_val, "prec_train": prec_train, "f2_val": f2_val}
    ecart = (np.mean(prec_train) - np.mean(prec_val)) / np.mean(prec_val) * 100
    print(f"{nom} : précision val {np.mean(prec_val) * 100:.1f} % (train {np.mean(prec_train) * 100:.1f} %, écart relatif {ecart:.0f} %) ; "
          f"score décisionnel F2 {np.mean(f2_val):.3f} ; {(time.time() - debut) / 60:.1f} min")

RandomForest profondeur libre, feuilles de 1 : précision val 26.7 % (train 100.0 %, écart relatif 275 %) ; score décisionnel F2 0.482 ; 0.1 min
RandomForest profondeur 20, feuilles de 5 : précision val 27.5 % (train 96.8 %, écart relatif 251 %) ; score décisionnel F2 0.486 ; 0.1 min
RandomForest profondeur 12, feuilles de 20 : précision val 28.0 % (train 48.0 %, écart relatif 71 %) ; score décisionnel F2 0.489 ; 0.1 min
CatBoost profondeur 8 : précision val 25.3 % (train 100.0 %, écart relatif 296 %) ; score décisionnel F2 0.471 ; 1.4 min
CatBoost profondeur 10 : précision val 24.9 % (train 100.0 %, écart relatif 302 %) ; score décisionnel F2 0.468 ; 2.7 min
MLP 256-128-64, sans arrêt anticipé : précision val 19.9 % (train 99.9 %, écart relatif 402 %) ; score décisionnel F2 0.428 ; 2.1 min
MLP 256-128-64, avec arrêt anticipé : précision val 26.0 % (train 36.8 %, écart relatif 42 %) ; score décisionnel F2 0.475 ; 0.3 min
MLP 128-64, sans arrêt anticipé : précision val 19.3 % (train 100.

## 3. Comparaison pli par pli avec `ml_11`

Pour chaque réglage poussé : son score décisionnel F2, pli par pli, comparé à celui du **même modèle** dans `ml_11` (réglé par la boucle, sous la règle de surapprentissage) et à celui du **meilleur modèle** de `ml_11`. Verdict comme dans les scénarios : « gain réel » si le gain moyen dépasse l'écart-type des gains sur les plis, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [5]:
from IPython.display import Markdown, display

reference = pd.read_csv(root_dir / "data" / "ML" / "probas_hors_pli" / "ml_11_v8.csv").set_index("ID").loc[train["ID"]]
assert (reference["dpnm"].to_numpy() == y).all(), "Probabilités de ml_11 non alignées sur le train"
modeles_ref = [c for c in reference.columns if c not in ("dpnm", "version_methode")]
f2_ref = {m: [f2_au_rappel_minimal(y[val], reference[m].to_numpy()[val]) for _, val in plis] for m in modeles_ref}
meilleur_ref = max(f2_ref, key=lambda m: np.mean(f2_ref[m]))

def verdict(gains):
    moyenne, ecart_type = np.mean(gains), np.std(gains)
    return "✅ gain réel" if moyenne > ecart_type else ("❌ perte" if moyenne < -ecart_type else "➖ dans le bruit")

lignes = []
for nom, m in mesures.items():
    gains_meme = np.array(m["f2_val"]) - np.array(f2_ref[m["modele_ml_11"]])
    gains_meilleur = np.array(m["f2_val"]) - np.array(f2_ref[meilleur_ref])
    lignes.append({
        "Réglage poussé": nom,
        "Précision val (%)": round(np.mean(m["prec_val"]) * 100, 1),
        "Précision train (%)": round(np.mean(m["prec_train"]) * 100, 1),
        "Écart relatif (%)": round((np.mean(m["prec_train"]) - np.mean(m["prec_val"])) / np.mean(m["prec_val"]) * 100),
        "Score décisionnel F2": round(np.mean(m["f2_val"]), 3),
        "Gain sur le même modèle de ml_11": f"{gains_meme.mean():+.3f} ± {gains_meme.std():.3f} ({(gains_meme > 0).sum()} plis sur 5)",
        "Verdict (même modèle)": verdict(gains_meme),
        f"Gain sur le meilleur de ml_11 ({meilleur_ref})": f"{gains_meilleur.mean():+.3f} ± {gains_meilleur.std():.3f} ({(gains_meilleur > 0).sum()} plis sur 5)",
        "Verdict (meilleur)": verdict(gains_meilleur),
    })
display(Markdown(f"**Référence** : `ml_11`, méthode v8 ; meilleur modèle : {meilleur_ref} (score décisionnel F2 {np.mean(f2_ref[meilleur_ref]):.3f})"))
display(pd.DataFrame(lignes).set_index("Réglage poussé"))

**Référence** : `ml_11`, méthode v8 ; meilleur modèle : LogisticRegression (score décisionnel F2 0.489)

,Précision val (%),Précision train (%),Écart relatif (%),Score décisionnel F2,Gain sur le même modèle de ml_11,Verdict (même modèle),Gain sur le meilleur de ml_11 (LogisticRegression),Verdict (meilleur)
Réglage poussé,,,,,,,,
"RandomForest profondeur libre, feuilles de 1",26.7,100.0,275,0.482,-0.001 ± 0.005 (1 plis sur 5),➖ dans le bruit,-0.007 ± 0.004 (0 plis sur 5),❌ perte
"RandomForest profondeur 20, feuilles de 5",27.5,96.8,251,0.486,+0.003 ± 0.005 (4 plis sur 5),➖ dans le bruit,-0.003 ± 0.003 (1 plis sur 5),➖ dans le bruit
"RandomForest profondeur 12, feuilles de 20",28.0,48.0,71,0.489,+0.006 ± 0.006 (4 plis sur 5),➖ dans le bruit,+0.000 ± 0.004 (3 plis sur 5),➖ dans le bruit
CatBoost profondeur 8,25.3,100.0,296,0.471,-0.015 ± 0.003 (0 plis sur 5),❌ perte,-0.018 ± 0.007 (0 plis sur 5),❌ perte
CatBoost profondeur 10,24.9,100.0,302,0.468,-0.018 ± 0.006 (0 plis sur 5),❌ perte,-0.021 ± 0.006 (0 plis sur 5),❌ perte
"MLP 256-128-64, sans arrêt anticipé",19.9,99.9,402,0.428,-0.051 ± 0.016 (0 plis sur 5),❌ perte,-0.061 ± 0.011 (0 plis sur 5),❌ perte
"MLP 256-128-64, avec arrêt anticipé",26.0,36.8,42,0.475,-0.004 ± 0.022 (3 plis sur 5),➖ dans le bruit,-0.014 ± 0.016 (1 plis sur 5),➖ dans le bruit
"MLP 128-64, sans arrêt anticipé",19.3,100.0,419,0.422,-0.057 ± 0.010 (0 plis sur 5),❌ perte,-0.067 ± 0.004 (0 plis sur 5),❌ perte


**Comment lire** : une ligne par réglage poussé. L'écart relatif montre à quel point le modèle a appris l'entraînement par cœur (au-delà de 5 %, la boucle l'aurait simplifié). Les deux colonnes de gain comparent son score décisionnel F2, pli par pli, au même modèle de `ml_11` puis au meilleur modèle de `ml_11`.

- **Un gain réel sur le meilleur modèle**, malgré un gros écart : la règle de surapprentissage bride ces modèles, et le plateau n'est pas une limite des données.
- **Rien de mieux que `ml_11`**, voire une perte : la complexité n'apporte rien, le plateau vient des données. La règle de surapprentissage ne coûte rien.